# Решение ДЗ — Модуль 01: Agentic RAG

LLM Zoomcamp 2026, DataTalksClub.

Ноутбук-версия решения. Q6 использует `toyaikit` (его `IPythonChatInterface` рассчитан именно на ноутбук).

**Перед запуском:**
```bash
uv add gitsource minsearch openai toyaikit python-dotenv
```
и положите `OPENAI_API_KEY` в `.env` рядом с ноутбуком.

`rag_helper.py` (рядом) уже адаптирован под схему `filename`/`content` и отдаёт usage.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from openai import OpenAI

from rag_helper import RAGBase

QUERY = "How does the agentic loop keep calling the model until it stops?"
openai_client = OpenAI()

## Preparation — тянем страницы уроков с коммита `8c1834d`

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
files = reader.read()

documents = [file.parse() for file in files]  # {'filename', 'content'}
documents[0]

## Q1. Сколько страниц уроков
Ожидаемый ответ: **72**.

In [ ]:
print("Q1:", len(documents))

## Q2. Индексация и поиск
`content` — text-поле, `filename` — keyword-поле. `filename` первого результата.
Ожидаемый ответ: **`01-agentic-rag/lessons/14-agentic-loop.md`**.

In [ ]:
def build_index(docs):
    index = Index(text_fields=["content"], keyword_fields=["filename"])
    index.fit(docs)
    return index

index = build_index(documents)
results = index.search(QUERY, num_results=5)
print("Q2:", results[0]["filename"])

## Q3. RAG — сколько входных (prompt) токенов
RAG поверх индекса из Q2 на `gpt-5.4-mini`. Ожидаемый ответ: **~7000**.

In [ ]:
rag = RAGBase(index=index, llm_client=openai_client)
res = rag.rag(QUERY)
print("Q3 input tokens:", res.input_tokens)
print(res.answer)

## Q4. Чанкинг
Sliding window `size=2000`, `step=1000`. Ожидаемый ответ: **~295**.

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)
print("Q4:", len(chunks))

## Q5. RAG с чанкингом
Тот же запрос, но индекс по чанкам. Сравниваем входные токены с Q3.
Ожидаемый ответ: **3× fewer**.

In [ ]:
chunk_index = build_index(chunks)
rag_chunked = RAGBase(index=chunk_index, llm_client=openai_client)
res_chunked = rag_chunked.rag(QUERY)
print("Q5 input tokens (chunked):", res_chunked.input_tokens)
print(f"Reduction: ~{res.input_tokens / res_chunked.input_tokens:.1f}x fewer")

## Q6. Превращаем в агента (toyaikit)
Инструмент `search` по чанковому индексу; агент сам решает, сколько раз искать.
Считаем число вызовов `search`. Ожидаемый ответ: **~4**.

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

def search(query: str) -> list[dict]:
    """Search the course lessons for chunks matching the given query."""
    return chunk_index.search(query, num_results=5)

agent_tools = Tools()
agent_tools.add_tool(search)  # схема выводится из type hint + docstring

instructions = (
    "You're a course teaching assistant. Answer the student's question "
    "using the search tool. Make multiple searches with different keywords "
    "before answering."
)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

In [ ]:
search_calls = sum(
    1 for m in result.all_messages
    if getattr(m, "type", None) == "function_call" and getattr(m, "name", None) == "search"
)
print("Q6 search calls:", search_calls)

> Если ячейка выше вернёт `0`, формат `all_messages` в вашей версии toyaikit иной.
> Посмотрите `result.all_messages` и/или используйте ручной цикл из `solution_var2.py` (Q6 там считается надёжно).